# Validação das cardinalidades entre entidades

## Introdução

Após a definição e validação dos relacionamentos conceituais, esta etapa analisa a quantidade mínima e máxima de ocorrências associadas entre as entidades do modelo.

A cardinalidade expressa quantas instâncias de uma entidade podem se relacionar com instâncias de outra. Sua definição deve combinar a semântica do domínio com o comportamento efetivamente observado no *Brazilian E-Commerce Public Dataset by Olist*.

Além da classificação geral em 1:1, 1:N ou N:N, esta análise observa a participação mínima encontrada nos dados, permitindo distinguir situações como 0..1, 1..1, 0..N e 1..N. A opcionalidade observada no dataset não substitui automaticamente a regra de negócio, mas constitui evidência importante para sua validação.

## Objetivos

- medir a multiplicidade observada em cada relacionamento aprovado;
- identificar relações 1:1, 1:N e N:N;
- verificar a existência de ocorrências sem associação;
- identificar relações cuja multiplicidade observada diverge das hipóteses iniciais;
- produzir evidências para a definição formal das cardinalidades no modelo conceitual;
- destacar relacionamentos N:N que exigirão tratamento específico na modelagem lógica.

## 1. Critérios de validação

Para cada relacionamento serão observados:

| Critério | Interpretação |
|---|---|
| Mínimo observado | Menor quantidade de ocorrências dependentes por instância de referência |
| Máximo observado | Maior quantidade de ocorrências dependentes por instância de referência |
| Ausência de associação | Quantidade de instâncias de referência com zero ocorrências dependentes |
| Ocorrência única | Quantidade de instâncias de referência com exatamente uma ocorrência dependente |
| Multiplicidade | Quantidade de instâncias de referência com mais de uma ocorrência dependente |
| Cardinalidade máxima | Classificação estrutural em 1:1, 1:N ou N:N |
| Opcionalidade observada | Evidência empírica de participação mínima, como 0..1, 1..1, 0..N ou 1..N |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

## 2. Localização e carregamento dos dados

In [2]:
data_directories = (
    Path("data/raw"),
    Path("../../data/raw"),
)

data_dir = next((path for path in data_directories if path.is_dir()), None)

if data_dir is None:
    raise FileNotFoundError(
        "Diretório data/raw não encontrado. Consulte data/README.md para obter os dados."
    )

data_dir.resolve()

PosixPath('/home/lucas/workspace/pessoal/ecommerce-analytics-data-model/data/raw')

In [3]:
arquivos_entidades = {
    "Clientes": "olist_customers_dataset.csv",
    "Pedidos": "olist_orders_dataset.csv",
    "Itens do Pedido": "olist_order_items_dataset.csv",
    "Produtos": "olist_products_dataset.csv",
    "Vendedores": "olist_sellers_dataset.csv",
    "Pagamentos": "olist_order_payments_dataset.csv",
    "Avaliações": "olist_order_reviews_dataset.csv",
    "Geolocalização": "olist_geolocation_dataset.csv",
}

entidades = {
    nome: pd.read_csv(data_dir / arquivo)
    for nome, arquivo in arquivos_entidades.items()
}

pd.DataFrame(
    {
        "Entidade": entidades.keys(),
        "Registros": [len(df) for df in entidades.values()],
        "Colunas": [len(df.columns) for df in entidades.values()],
    }
)

,Entidade,Registros,Colunas
0,Clientes,99441,5
1,Pedidos,99441,8
2,Itens do Pedido,112650,7
3,Produtos,32951,9
4,Vendedores,3095,4
5,Pagamentos,103886,5
6,Avaliações,99224,7
7,Geolocalização,1000163,5


## 3. Relacionamentos submetidos à análise

Os oito relacionamentos previamente aprovados são:

1. Cliente realiza Pedido;
2. Pedido possui Item do Pedido;
3. Item do Pedido refere-se a Produto;
4. Vendedor vende Item do Pedido;
5. Pedido possui Pagamento;
6. Pedido recebe Avaliação;
7. Cliente associa-se geograficamente a Geolocalização;
8. Vendedor associa-se geograficamente a Geolocalização.

A análise é dividida entre relacionamentos baseados em identificadores diretos e associações geográficas mediadas por prefixo de CEP.

In [4]:
relacionamentos_diretos = [
    {
        "relacionamento": "Cliente realiza Pedido",
        "lado_a_entidade": "Clientes",
        "lado_a_coluna": "customer_id",
        "lado_b_entidade": "Pedidos",
        "lado_b_coluna": "customer_id",
    },
    {
        "relacionamento": "Pedido possui Item do Pedido",
        "lado_a_entidade": "Pedidos",
        "lado_a_coluna": "order_id",
        "lado_b_entidade": "Itens do Pedido",
        "lado_b_coluna": "order_id",
    },
    {
        "relacionamento": "Item do Pedido refere-se a Produto",
        "lado_a_entidade": "Produtos",
        "lado_a_coluna": "product_id",
        "lado_b_entidade": "Itens do Pedido",
        "lado_b_coluna": "product_id",
    },
    {
        "relacionamento": "Vendedor vende Item do Pedido",
        "lado_a_entidade": "Vendedores",
        "lado_a_coluna": "seller_id",
        "lado_b_entidade": "Itens do Pedido",
        "lado_b_coluna": "seller_id",
    },
    {
        "relacionamento": "Pedido possui Pagamento",
        "lado_a_entidade": "Pedidos",
        "lado_a_coluna": "order_id",
        "lado_b_entidade": "Pagamentos",
        "lado_b_coluna": "order_id",
    },
    {
        "relacionamento": "Pedido recebe Avaliação",
        "lado_a_entidade": "Pedidos",
        "lado_a_coluna": "order_id",
        "lado_b_entidade": "Avaliações",
        "lado_b_coluna": "order_id",
    },
]

pd.DataFrame(relacionamentos_diretos)

,relacionamento,lado_a_entidade,lado_a_coluna,lado_b_entidade,lado_b_coluna
0,Cliente realiza Pedido,Clientes,customer_id,Pedidos,customer_id
1,Pedido possui Item do Pedido,Pedidos,order_id,Itens do Pedido,order_id
2,Item do Pedido refere-se a Produto,Produtos,product_id,Itens do Pedido,product_id
3,Vendedor vende Item do Pedido,Vendedores,seller_id,Itens do Pedido,seller_id
4,Pedido possui Pagamento,Pedidos,order_id,Pagamentos,order_id
5,Pedido recebe Avaliação,Pedidos,order_id,Avaliações,order_id


## 4. Funções de análise de multiplicidade

Para cada relacionamento direto, a análise é realizada nos dois sentidos:

- quantas ocorrências de B existem para cada instância de A;
- quantas ocorrências de A existem para cada instância de B.

Isso permite distinguir corretamente relações 1:1, 1:N e N:N.

In [5]:
def distribuicao_associacoes(
    referencia: pd.DataFrame,
    coluna_referencia: str,
    dependente: pd.DataFrame,
    coluna_dependente: str,
):
    chaves_referencia = pd.Index(
        referencia[coluna_referencia].dropna().unique()
    )

    contagens_dependentes = (
        dependente[coluna_dependente]
        .dropna()
        .value_counts()
        .reindex(chaves_referencia, fill_value=0)
    )

    return contagens_dependentes


def resumir_distribuicao(contagens: pd.Series) -> dict:
    if len(contagens) == 0:
        return {
            "mínimo": np.nan,
            "mediana": np.nan,
            "média": np.nan,
            "máximo": np.nan,
            "zero": 0,
            "um": 0,
            "mais_de_um": 0,
        }

    return {
        "mínimo": int(contagens.min()),
        "mediana": float(contagens.median()),
        "média": float(contagens.mean()),
        "máximo": int(contagens.max()),
        "zero": int((contagens == 0).sum()),
        "um": int((contagens == 1).sum()),
        "mais_de_um": int((contagens > 1).sum()),
    }


def classificar_cardinalidade(max_a_para_b: int, max_b_para_a: int) -> str:
    if max_a_para_b <= 1 and max_b_para_a <= 1:
        return "1:1"
    if max_a_para_b > 1 and max_b_para_a <= 1:
        return "1:N"
    if max_a_para_b <= 1 and max_b_para_a > 1:
        return "N:1"
    return "N:N"


def classificar_opcionalidade(minimo: int, maximo: int) -> str:
    if pd.isna(minimo) or pd.isna(maximo):
        return "Indeterminada"
    if maximo <= 1:
        return "0..1" if minimo == 0 else "1..1"
    return "0..N" if minimo == 0 else "1..N"

## 5. Cardinalidades dos relacionamentos diretos

In [6]:
resultados_cardinalidade = []
detalhes_distribuicoes = {}

for rel in relacionamentos_diretos:
    a = entidades[rel["lado_a_entidade"]]
    b = entidades[rel["lado_b_entidade"]]

    a_para_b = distribuicao_associacoes(
        referencia=a,
        coluna_referencia=rel["lado_a_coluna"],
        dependente=b,
        coluna_dependente=rel["lado_b_coluna"],
    )

    b_para_a = distribuicao_associacoes(
        referencia=b,
        coluna_referencia=rel["lado_b_coluna"],
        dependente=a,
        coluna_dependente=rel["lado_a_coluna"],
    )

    resumo_a_b = resumir_distribuicao(a_para_b)
    resumo_b_a = resumir_distribuicao(b_para_a)

    resultados_cardinalidade.append(
        {
            "Relacionamento": rel["relacionamento"],
            "A": rel["lado_a_entidade"],
            "B": rel["lado_b_entidade"],
            "Máx. B por A": resumo_a_b["máximo"],
            "Máx. A por B": resumo_b_a["máximo"],
            "Cardinalidade observada": classificar_cardinalidade(
                resumo_a_b["máximo"], resumo_b_a["máximo"]
            ),
            "Participação de A": classificar_opcionalidade(
                resumo_a_b["mínimo"], resumo_a_b["máximo"]
            ),
            "Participação de B": classificar_opcionalidade(
                resumo_b_a["mínimo"], resumo_b_a["máximo"]
            ),
            "A sem B": resumo_a_b["zero"],
            "A com 1 B": resumo_a_b["um"],
            "A com >1 B": resumo_a_b["mais_de_um"],
            "B sem A": resumo_b_a["zero"],
            "B com 1 A": resumo_b_a["um"],
            "B com >1 A": resumo_b_a["mais_de_um"],
        }
    )

    detalhes_distribuicoes[rel["relacionamento"]] = {
        "A_para_B": a_para_b,
        "B_para_A": b_para_a,
        "resumo_A_para_B": resumo_a_b,
        "resumo_B_para_A": resumo_b_a,
    }

df_cardinalidades = pd.DataFrame(resultados_cardinalidade)
df_cardinalidades

,Relacionamento,A,B,Máx. B por A,Máx. A por B,Cardinalidade observada,Participação de A,Participação de B,A sem B,A com 1 B,A com >1 B,B sem A,B com 1 A,B com >1 A
0,Cliente realiza Pedido,Clientes,Pedidos,1,1,1:1,1..1,1..1,0,99441,0,0,99441,0
1,Pedido possui Item do Pedido,Pedidos,Itens do Pedido,21,1,1:N,0..N,1..1,775,88863,9803,0,98666,0
2,Item do Pedido refere-se a Produto,Produtos,Itens do Pedido,527,1,1:N,1..N,1..1,0,18117,14834,0,32951,0
3,Vendedor vende Item do Pedido,Vendedores,Itens do Pedido,2033,1,1:N,1..N,1..1,0,509,2586,0,3095,0
4,Pedido possui Pagamento,Pedidos,Pagamentos,29,1,1:N,0..N,1..1,1,96479,2961,0,99440,0
5,Pedido recebe Avaliação,Pedidos,Avaliações,3,1,1:N,0..N,1..1,768,98126,547,0,98673,0


### 5.1 Estatísticas detalhadas

A tabela abaixo apresenta mínimo, mediana, média e máximo de ocorrências para cada sentido do relacionamento.

In [7]:
linhas_detalhes = []

for rel in relacionamentos_diretos:
    nome = rel["relacionamento"]
    det = detalhes_distribuicoes[nome]

    for sentido, resumo in [
        (f"{rel['lado_a_entidade']} → {rel['lado_b_entidade']}", det["resumo_A_para_B"]),
        (f"{rel['lado_b_entidade']} → {rel['lado_a_entidade']}", det["resumo_B_para_A"]),
    ]:
        linhas_detalhes.append(
            {
                "Relacionamento": nome,
                "Sentido": sentido,
                "Mínimo": resumo["mínimo"],
                "Mediana": resumo["mediana"],
                "Média": resumo["média"],
                "Máximo": resumo["máximo"],
                "Zero ocorrências": resumo["zero"],
                "Uma ocorrência": resumo["um"],
                "Mais de uma": resumo["mais_de_um"],
            }
        )

df_detalhes = pd.DataFrame(linhas_detalhes)
df_detalhes

,Relacionamento,Sentido,Mínimo,Mediana,Média,Máximo,Zero ocorrências,Uma ocorrência,Mais de uma
0,Cliente realiza Pedido,Clientes → Pedidos,1,1.0,1.000000,1,0,99441,0
1,Cliente realiza Pedido,Pedidos → Clientes,1,1.0,1.000000,1,0,99441,0
2,Pedido possui Item do Pedido,Pedidos → Itens do Pedido,0,1.0,1.132833,21,775,88863,9803
3,Pedido possui Item do Pedido,Itens do Pedido → Pedidos,1,1.0,1.000000,1,0,98666,0
4,Item do Pedido refere-se a Produto,Produtos → Itens do Pedido,1,1.0,3.418713,527,0,18117,14834
5,Item do Pedido refere-se a Produto,Itens do Pedido → Produtos,1,1.0,1.000000,1,0,32951,0
6,Vendedor vende Item do Pedido,Vendedores → Itens do Pedido,1,8.0,36.397415,2033,0,509,2586
7,Vendedor vende Item do Pedido,Itens do Pedido → Vendedores,1,1.0,1.000000,1,0,3095,0
8,Pedido possui Pagamento,Pedidos → Pagamentos,0,1.0,1.044700,29,1,96479,2961
9,Pedido possui Pagamento,Pagamentos → Pedidos,1,1.0,1.000000,1,0,99440,0


## 6. Casos que exigem atenção especial

### 6.1 Cliente — Pedido

A distinção entre `customer_id` e `customer_unique_id` é essencial.

A entidade Cliente atualmente adotada no modelo utiliza `customer_id` como identificador conceitual. O mesmo consumidor persistente pode aparecer em múltiplos registros de Cliente por meio de diferentes valores de `customer_id`.

A análise abaixo compara:

- quantidade de pedidos por `customer_id`;
- quantidade de pedidos por `customer_unique_id`.

Isso permite evitar a interpretação incorreta de que a cardinalidade da entidade Cliente atual é necessariamente a mesma que a cardinalidade do consumidor persistente.

In [8]:
clientes = entidades["Clientes"]
pedidos = entidades["Pedidos"]

pedidos_com_cliente_unico = pedidos.merge(
    clientes[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

resumo_customer_id = (
    pedidos["customer_id"]
    .value_counts()
    .describe()
)

resumo_customer_unique = (
    pedidos_com_cliente_unico["customer_unique_id"]
    .value_counts()
    .describe()
)

display(
    pd.DataFrame(
        {
            "customer_id": resumo_customer_id,
            "customer_unique_id": resumo_customer_unique,
        }
    )
)

,customer_id,customer_unique_id
count,99441.0,96096.000000
mean,1.0,1.034809
std,0.0,0.214384
min,1.0,1.000000
25%,1.0,1.000000
50%,1.0,1.000000
75%,1.0,1.000000
max,1.0,17.000000


In [9]:
pd.DataFrame(
    {
        "Métrica": [
            "Clientes por customer_id com mais de um pedido",
            "Consumidores por customer_unique_id com mais de um pedido",
            "Máximo de pedidos por customer_id",
            "Máximo de pedidos por customer_unique_id",
        ],
        "Valor": [
            int((pedidos["customer_id"].value_counts() > 1).sum()),
            int((pedidos_com_cliente_unico["customer_unique_id"].value_counts() > 1).sum()),
            int(pedidos["customer_id"].value_counts().max()),
            int(pedidos_com_cliente_unico["customer_unique_id"].value_counts().max()),
        ],
    }
)

,Métrica,Valor
0,Clientes por customer_id com mais de um pedido,0
1,Consumidores por customer_unique_id com mais de um pedido,2997
2,Máximo de pedidos por customer_id,1
3,Máximo de pedidos por customer_unique_id,17


### 6.2 Pedido — Avaliação

A hipótese preliminar de relacionamento 1:1 deve ser confrontada com os dados, pois `order_id` não é único na entidade Avaliação.

A análise abaixo identifica quantos pedidos possuem zero, uma ou múltiplas avaliações registradas.

In [10]:
avaliacoes_por_pedido = (
    entidades["Avaliações"]["order_id"]
    .value_counts()
    .reindex(entidades["Pedidos"]["order_id"].unique(), fill_value=0)
)

pd.DataFrame(
    {
        "Situação": [
            "Pedidos sem avaliação",
            "Pedidos com exatamente uma avaliação",
            "Pedidos com múltiplas avaliações",
            "Máximo de avaliações em um pedido",
        ],
        "Quantidade": [
            int((avaliacoes_por_pedido == 0).sum()),
            int((avaliacoes_por_pedido == 1).sum()),
            int((avaliacoes_por_pedido > 1).sum()),
            int(avaliacoes_por_pedido.max()),
        ],
    }
)

,Situação,Quantidade
0,Pedidos sem avaliação,768
1,Pedidos com exatamente uma avaliação,98126
2,Pedidos com múltiplas avaliações,547
3,Máximo de avaliações em um pedido,3


## 7. Cardinalidade das associações geográficas

As relações envolvendo Geolocalização não podem ser analisadas como simples correspondências de chave estrangeira, pois o atributo de associação é o prefixo de CEP e esse valor se repete em ambas as extremidades.

Para cada prefixo, são observados:

- quantidade de Clientes associados;
- quantidade de Vendedores associados;
- quantidade de registros de Geolocalização associados.

Quando um mesmo prefixo está presente em múltiplas instâncias dos dois lados, a estrutura bruta caracteriza uma associação potencialmente N:N.

In [11]:
geo = entidades["Geolocalização"]
clientes = entidades["Clientes"]
vendedores = entidades["Vendedores"]

clientes_por_cep = clientes["customer_zip_code_prefix"].value_counts()
vendedores_por_cep = vendedores["seller_zip_code_prefix"].value_counts()
geo_por_cep = geo["geolocation_zip_code_prefix"].value_counts()

ceps_clientes_geo = clientes_por_cep.index.intersection(geo_por_cep.index)
ceps_vendedores_geo = vendedores_por_cep.index.intersection(geo_por_cep.index)

analise_cliente_geo = pd.DataFrame(
    {
        "clientes_no_prefixo": clientes_por_cep.reindex(ceps_clientes_geo),
        "geolocalizacoes_no_prefixo": geo_por_cep.reindex(ceps_clientes_geo),
    }
)

analise_vendedor_geo = pd.DataFrame(
    {
        "vendedores_no_prefixo": vendedores_por_cep.reindex(ceps_vendedores_geo),
        "geolocalizacoes_no_prefixo": geo_por_cep.reindex(ceps_vendedores_geo),
    }
)

display(analise_cliente_geo.describe())
display(analise_vendedor_geo.describe())

,clientes_no_prefixo,geolocalizacoes_no_prefixo
count,14837.000000,14837.000000
mean,6.683494,65.262452
std,8.408868,76.840344
min,1.000000,1.000000
25%,2.000000,21.000000
50%,4.000000,42.000000
75%,8.000000,82.000000
max,142.000000,1146.000000


,vendedores_no_prefixo,geolocalizacoes_no_prefixo
count,2239.000000,2239.000000
mean,1.379187,120.060741
std,1.296360,111.703756
min,1.000000,1.000000
25%,1.000000,46.000000
50%,1.000000,89.000000
75%,1.000000,152.500000
max,49.000000,965.000000


In [12]:
resumo_geo = pd.DataFrame(
    {
        "Relacionamento": [
            "Cliente — Geolocalização",
            "Vendedor — Geolocalização",
        ],
        "Prefixos compartilhados": [
            len(ceps_clientes_geo),
            len(ceps_vendedores_geo),
        ],
        "Prefixos com múltiplos registros no lado de negócio": [
            int((analise_cliente_geo["clientes_no_prefixo"] > 1).sum()),
            int((analise_vendedor_geo["vendedores_no_prefixo"] > 1).sum()),
        ],
        "Prefixos com múltiplas geolocalizações": [
            int((analise_cliente_geo["geolocalizacoes_no_prefixo"] > 1).sum()),
            int((analise_vendedor_geo["geolocalizacoes_no_prefixo"] > 1).sum()),
        ],
        "Prefixos com multiplicidade nos dois lados": [
            int(
                (
                    (analise_cliente_geo["clientes_no_prefixo"] > 1)
                    & (analise_cliente_geo["geolocalizacoes_no_prefixo"] > 1)
                ).sum()
            ),
            int(
                (
                    (analise_vendedor_geo["vendedores_no_prefixo"] > 1)
                    & (analise_vendedor_geo["geolocalizacoes_no_prefixo"] > 1)
                ).sum()
            ),
        ],
    }
)

resumo_geo

,Relacionamento,Prefixos compartilhados,Prefixos com múltiplos registros no lado de negócio,Prefixos com múltiplas geolocalizações,Prefixos com multiplicidade nos dois lados
0,Cliente — Geolocalização,14837,11935,14714,11914
1,Vendedor — Geolocalização,2239,537,2230,537


## 8. Síntese para decisão conceitual

A tabela produzida nesta seção deve ser interpretada em conjunto com as regras de negócio.

### Orientação

- `1:1`: nenhuma das extremidades apresenta multiplicidade superior a 1;
- `1:N`: uma instância de A pode se associar a múltiplas instâncias de B, enquanto cada B se associa a no máximo uma A;
- `N:1`: representação inversa de 1:N;
- `N:N`: ambas as extremidades apresentam multiplicidade superior a 1;
- participação mínima igual a zero indica opcionalidade **observada** no dataset, não necessariamente uma regra definitiva do domínio.

A classificação conceitual final deve registrar explicitamente qualquer divergência entre a estrutura empírica e a regra de negócio.

In [13]:
df_sintese = df_cardinalidades[
    [
        "Relacionamento",
        "Cardinalidade observada",
        "Participação de A",
        "Participação de B",
        "A sem B",
        "A com >1 B",
        "B sem A",
        "B com >1 A",
    ]
].copy()

df_sintese

,Relacionamento,Cardinalidade observada,Participação de A,Participação de B,A sem B,A com >1 B,B sem A,B com >1 A
0,Cliente realiza Pedido,1:1,1..1,1..1,0,0,0,0
1,Pedido possui Item do Pedido,1:N,0..N,1..1,775,9803,0,0
2,Item do Pedido refere-se a Produto,1:N,1..N,1..1,0,14834,0,0
3,Vendedor vende Item do Pedido,1:N,1..N,1..1,0,2586,0,0
4,Pedido possui Pagamento,1:N,0..N,1..1,1,2961,0,0
5,Pedido recebe Avaliação,1:N,0..N,1..1,768,547,0,0


## 9. Questões para validação final

Após a execução do notebook, responder:

1. `Cliente realiza Pedido` é 1:1 quando Cliente é identificado por `customer_id`?
2. O comportamento de `customer_unique_id` confirma que consumidor persistente e registro de Cliente são conceitos diferentes?
3. Todo Pedido possui pelo menos um Item do Pedido no dataset? Se não, a ausência é compatível com estados incompletos ou cancelados?
4. Cada Item do Pedido referencia exatamente um Produto?
5. Cada Item do Pedido está associado a exatamente um Vendedor?
6. Pedido — Pagamento apresenta multiplicidade 1:N? Existem pedidos sem registros de pagamento?
7. Pedido — Avaliação é efetivamente 1:N ou a multiplicidade superior a 1 decorre de duplicidades/anomalias que exigem interpretação?
8. Cliente — Geolocalização e Vendedor — Geolocalização configuram N:N na estrutura bruta mediada por prefixo de CEP?
9. Alguma cardinalidade observada contradiz as regras de negócio previamente documentadas?
10. Quais relações N:N deverão ser destacadas para futura resolução no modelo lógico?

## 10. Resultado da etapa

A decisão formal das cardinalidades deve ser registrada somente após a execução e interpretação dos resultados.

O documento de Modelagem Conceitual deverá incorporar:

- cardinalidade final de cada relacionamento;
- participação mínima quando metodologicamente justificável;
- divergências entre regra de negócio e comportamento observado;
- justificativa para eventuais relações N:N;
- observações necessárias para a transformação posterior em modelo lógico.

A cardinalidade deve refletir o domínio representado pelo projeto, utilizando o dataset como evidência empírica e não como única fonte normativa.